# Phase 3, experiment 11: a graph neural network (GraphSAGE)

**Why this experiment:** Hand-made neighbour features (notebook 06) and score smoothing (notebook 11)
did not help. A graph neural network (GNN) learns how to combine a transaction's own features with its
neighbours' features, and it is trained directly on the labels. All transactions, labeled or not, take
part in passing information along the edges; only the labeled training rows count towards the training
loss, so no validation or test labels are used.

**Decision:** try a GNN, with a firm cut-off: if by 18 Oct it does not raise the fold mean
without lowering fold 2 (alone or blended with LightGBM), drop it and start finalising.

**The model: GraphSAGE with separate directions.** Each GraphSAGE layer computes a new vector for every
transaction from (a) its own current vector and (b) the mean of its neighbours' vectors, each multiplied
by learned weights. Here every layer has two parts that are added together:

- one that looks at **predecessors** (transactions whose outputs this one spends), and
- one that looks at **successors** (transactions that spend this one's outputs),

because money coming in and going out can mean different things (the same idea as notebook 06). Two
layers are stacked, so each transaction sees its neighbours and its neighbours' neighbours. A final
linear layer turns the vector into one score.

**Starting settings (chosen as common defaults, not tuned):**

| Setting | Value | Meaning |
|---|---|---|
| hidden size | 64 | length of each transaction's vector inside the network |
| layers | 2 | how many hops of neighbours are seen |
| dropout | 0.3 | share of the vector randomly set to zero while training, to reduce overfitting |
| optimiser | Adam, learning rate 0.01, weight decay 0.0005 | how the weights are updated |
| epochs | 200 | how many full passes over the training rows |
| class imbalance | `pos_weight` = licit rows / illicit rows | an illicit row counts as much as about 9 licit rows in the loss |

**Transactions outside train and test:** (46,668 of them) have no features, so they are left out, and so
are the edges that touch them.

**Why LightGBM is not in this notebook:** On this Mac, training LightGBM in the same Python session
as PyTorch crashes the kernel (both libraries bring their own copy of the OpenMP threading library). So
this notebook only trains the GNN and saves its predictions to `../outputs/`. Notebook
`14_gnn_blend.ipynb` loads them and compares and blends them with LightGBM.

Validation: the three expanding-window folds. Run time: about 6 to 8 minutes.

## 1. Load and build the graph

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"
OUT_DIR = "../outputs"
os.makedirs(OUT_DIR, exist_ok=True)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
raw_cols = [c for c in train.columns if c.startswith("feat_")]

# One table of nodes: every train and test transaction. Test rows have no label (NaN).
nodes = pd.concat([train[["txId", "time_step", "label"] + raw_cols],
                   test[["txId", "time_step"] + raw_cols]], ignore_index=True)

# PyTorch Geometric refers to nodes by their row number (0, 1, 2, ...), not by txId
row_of = pd.Series(np.arange(len(nodes)), index=nodes["txId"])

# Keep only edges whose two ends are both in train or test
known = edges["txId1"].isin(row_of.index) & edges["txId2"].isin(row_of.index)
kept = edges[known]

# edge_index has 2 rows: source row numbers and destination row numbers (predecessor -> successor)
edge_index = torch.tensor(np.vstack([row_of[kept["txId1"]].values, row_of[kept["txId2"]].values]),
                          dtype=torch.long)

time_step = nodes["time_step"].values
label = nodes["label"].values          # NaN for unlabeled and test rows
print("nodes:", len(nodes), " edges kept:", edge_index.shape[1], "of", len(edges))

/Users/dorislai/Downloads/bt4012_kaggle_competition/bt4012-kaggle/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


nodes: 157101  edges kept: 175918 of 234355


## 2. The model

In [ ]:
class DirectedSAGE(torch.nn.Module):

    def __init__(self, n_features, hidden):
        super().__init__()
        self.pred1 = SAGEConv(n_features, hidden)   # layer 1, looking at predecessors
        self.succ1 = SAGEConv(n_features, hidden)   # layer 1, looking at successors
        self.pred2 = SAGEConv(hidden, hidden)       # layer 2, predecessors
        self.succ2 = SAGEConv(hidden, hidden)       # layer 2, successors
        self.out = torch.nn.Linear(hidden, 1)       # vector -> one score

    def forward(self, x, edge_index):
        # SAGEConv passes information from source to destination, so edge_index gives each node
        # its predecessors. Flipping the two rows reverses every edge, which gives its successors.
        reversed_edges = edge_index.flip(0)

        h = F.relu(self.pred1(x, edge_index) + self.succ1(x, reversed_edges))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.pred2(h, edge_index) + self.succ2(h, reversed_edges))
        h = F.dropout(h, p=0.3, training=self.training)
        return self.out(h).squeeze(-1)      # one raw score (logit) per node

## 3. Training function

For one time period, this:

1. scales the features using the mean and standard deviation of the **training-period rows only**
   (all rows, labeled or not, since no labels are involved), and clips extreme values to -10..10;
2. trains the network for 200 epochs on the labeled training-period rows;
3. returns the predicted probability for every node.

The whole graph is used in every epoch ("full-batch" training), which is possible because it is small.

In [ ]:
EPOCHS = 200
HIDDEN = 64

def train_gnn(train_last_step, check_rows=None, check_labels=None):
    # Train on labeled rows with time_step <= train_last_step. Return a probability for every node.
    # If check_rows/check_labels are given, print their AUC every 25 epochs
    in_train_period = time_step <= train_last_step

    # Feature scaling fitted on the training period only
    mean = nodes.loc[in_train_period, raw_cols].mean()
    std = nodes.loc[in_train_period, raw_cols].std().replace(0, 1)
    scaled = ((nodes[raw_cols] - mean) / std).clip(-10, 10)
    x = torch.tensor(scaled.values, dtype=torch.float32)

    # Rows that count towards the loss: labeled rows in the training period
    train_rows = np.where(in_train_period & ~np.isnan(label))[0]
    y = torch.tensor(label[train_rows], dtype=torch.float32)
    pos_weight = (y == 0).sum() / (y == 1).sum()      

    torch.manual_seed(SEED)                           # same starting weights every run
    model = DirectedSAGE(len(raw_cols), HIDDEN)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, edge_index)
        loss = F.binary_cross_entropy_with_logits(logits[train_rows], y, pos_weight=pos_weight)
        loss.backward()
        optimizer.step()

        if check_rows is not None and epoch % 25 == 0:
            model.eval()
            with torch.no_grad():
                p = torch.sigmoid(model(x, edge_index)).numpy()
            print(f"  epoch {epoch}: loss {loss.item():.4f}, validation AUC {roc_auc_score(check_labels, p[check_rows]):.4f}")

    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(x, edge_index)).numpy()

## 4. Score the GNN on the three folds

In [ ]:
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]

fold_predictions = []   
gnn_auc = {}

for fold in FOLDS:
    in_valid = (time_step > fold["train_last"]) & (time_step <= fold["valid_last"]) & ~np.isnan(label)
    valid_rows = np.where(in_valid)[0]
    valid_labels = label[valid_rows]

    print(fold["name"])
    probs = train_gnn(fold["train_last"], valid_rows, valid_labels)
    gnn_auc[fold["name"]] = roc_auc_score(valid_labels, probs[valid_rows])

    fold_predictions.append(pd.DataFrame({
        "fold": fold["name"],
        "txId": nodes["txId"].values[valid_rows],
        "gnn_score": probs[valid_rows],
    }))

pd.concat(fold_predictions).to_csv(f"{OUT_DIR}/gnn_fold_predictions.csv", index=False)

fold 1
  epoch 25: loss 0.0809, validation AUC 0.9557
  epoch 50: loss 0.0259, validation AUC 0.9547
  epoch 75: loss 0.0206, validation AUC 0.9562
  epoch 100: loss 0.0134, validation AUC 0.9533
  epoch 125: loss 0.0199, validation AUC 0.9573
  epoch 150: loss 0.0146, validation AUC 0.9555
  epoch 175: loss 0.0150, validation AUC 0.9566
  epoch 200: loss 0.0162, validation AUC 0.9562
fold 2
  epoch 25: loss 0.1352, validation AUC 0.9095
  epoch 50: loss 0.0488, validation AUC 0.9155
  epoch 75: loss 0.0215, validation AUC 0.9100
  epoch 100: loss 0.0226, validation AUC 0.9113
  epoch 125: loss 0.0178, validation AUC 0.9067
  epoch 150: loss 0.0188, validation AUC 0.9050
  epoch 175: loss 0.0149, validation AUC 0.9086
  epoch 200: loss 0.0182, validation AUC 0.9065
fold 3
  epoch 25: loss 0.1837, validation AUC 0.9752
  epoch 50: loss 0.0753, validation AUC 0.9761
  epoch 75: loss 0.0389, validation AUC 0.9747
  epoch 100: loss 0.0324, validation AUC 0.9803
  epoch 125: loss 0.0291, va

In [ ]:
# Default LightGBM on the raw features from notebook 07, for comparison
lgbm_auc = {"fold 1": 0.9888, "fold 2": 0.9572, "fold 3": 0.9965}

comparison = pd.DataFrame({"GNN": gnn_auc, "LightGBM (notebook 07)": lgbm_auc}).T
comparison["mean"] = comparison[["fold 1", "fold 2", "fold 3"]].mean(axis=1)
comparison.round(4)

,fold 1,fold 2,fold 3,mean
GNN,0.9562,0.9065,0.9807,0.9478
LightGBM (notebook 07),0.9888,0.9572,0.9965,0.9808


## 5. Train on all labeled rows and predict the test set

The GNN is retrained on steps 1 to 35 and its test predictions are saved, so notebook 14 can build a
blended submission if the blend helps on the folds.

In [6]:
final_probs = train_gnn(train_last_step=35)
is_test = np.isin(nodes["txId"].values, test["txId"].values)
pd.DataFrame({"txId": nodes["txId"].values[is_test], "gnn_score": final_probs[is_test]}) \
    .to_csv(f"{OUT_DIR}/gnn_test_predictions.csv", index=False)
print("Saved test predictions for", is_test.sum(), "test transactions")

Saved test predictions for 15329 test transactions


## Findings

- **How does the GNN alone compare with LightGBM on each fold?** It is clearly weaker on every fold:
  0.9562 against 0.9888 on fold 1, 0.9065 against 0.9572 on fold 2 and 0.9807 against 0.9965 on fold 3,
  so a mean of 0.9478 against 0.9808. The gap is largest on fold 2 (0.051), the fold with the new kind of
  illicit transaction.
- **What happens on fold 2 / step 24?** The GNN scores 0.653 on step 24 (see notebook 14), against 0.787
  for LightGBM. Learning from neighbours did not help with this pattern, which fits notebook 05: most of
  the missed illicit transactions are connected to other transactions that look just as unfamiliar.
- **Does the validation AUC change much over the epochs?** Not much after the first 25 epochs. It moves
  within about 0.004 on fold 1 (0.953 to 0.957), 0.011 on fold 2 (0.905 to 0.916) and 0.006 on fold 3
  (0.975 to 0.981), while the training loss keeps falling. Training for longer is unlikely to change the
  result. These settings were not tuned, but the gap to LightGBM is large.

